# 第 10 章习题与解答

> 本章习题围绕 LoRA 的参数量计算、方形过滤设计决策、零初始化原理展开。

## Exercise 10.1（易）

**题目**:手算 LoRA(rank=16)在 768×768 的 `o_proj` 上的参数量,对比原始参数量。压缩比是多少?

<details><summary><b>参考答案</b></summary>

In [ ]:
# 手算 + 代码验证

d = 768   # hidden_size
r = 16    # LoRA rank

# 原始 o_proj: 一个 768×768 的矩阵(无 bias)
original = d * d
print(f"原始 o_proj 参数量: {d} × {d} = {original:,}")

# LoRA: A(r×d) + B(d×r)
A_params = r * d      # 降维矩阵 A: 16×768
B_params = d * r      # 升维矩阵 B: 768×16
lora_total = A_params + B_params
print(f"LoRA A 参数量: {r} × {d} = {A_params:,}")
print(f"LoRA B 参数量: {d} × {r} = {B_params:,}")
print(f"LoRA 合计: {lora_total:,}")

# 压缩比
ratio = original / lora_total
print(f"\n压缩比: {original:,} / {lora_total:,} = {ratio:.1f}×")
print(f"LoRA 占原始参数的: {lora_total / original * 100:.2f}%")

**解析**:

原始 `o_proj` 是 `nn.Linear(768, 768, bias=False)`,参数量 = $768 \times 768 = 589{,}824$。

LoRA(rank=16)引入两个小矩阵:
- $A \in \mathbb{R}^{16 \times 768}$:$16 \times 768 = 12{,}288$
- $B \in \mathbb{R}^{768 \times 16}$:$768 \times 16 = 12{,}288$
- 合计:$24{,}576$

压缩比 = $589{,}824 / 24{,}576 = 24\times$,即 LoRA 只用原始参数量的 **4.17%**。

> 注意:这只是单个模块的压缩比。整个模型 16 个模块(q_proj + o_proj × 8 层)的 LoRA 参数总计 393,216,占 63.9M 总参数的 0.61%。

</details>

## Exercise 10.2（中）

**题目**:minimind 的 `apply_lora` 用 `in_features == out_features` 过滤。在标准配置下(hidden_size=768, num_attention_heads=8, num_kv_heads=4, head_dim=96),哪些 proj 被挂 LoRA?为什么 k/v_proj 不被挂?如果去掉这个过滤条件,全部 Linear 都挂会怎样?

<details><summary><b>参考答案</b></summary>

In [ ]:
import torch.nn as nn
from model.model_minimind import MiniMindConfig, MiniMindForCausalLM

config = MiniMindConfig()

# 计算每个 proj 的维度
h = config.hidden_size           # 768
n_heads = config.num_attention_heads  # 8
n_kv = config.num_key_value_heads     # 4
hd = config.head_dim             # 96

print(f"hidden_size = {h}, num_heads = {n_heads}, num_kv_heads = {n_kv}, head_dim = {hd}")
print(f"\n=== 各 proj 维度 ===")
print(f"q_proj: {h} → {n_heads}×{hd} = {n_heads*hd}  方形: {h == n_heads*hd}")
print(f"k_proj: {h} → {n_kv}×{hd} = {n_kv*hd}    方形: {h == n_kv*hd}")
print(f"v_proj: {h} → {n_kv}×{hd} = {n_kv*hd}    方形: {h == n_kv*hd}")
print(f"o_proj: {n_heads}×{hd} → {h} = {h}  方形: {n_heads*hd == h}")

print(f"\n=== 被挂 LoRA 的模块 ===")
model = MiniMindForCausalLM(config)
from model.model_lora import apply_lora
apply_lora(model)
for name, m in model.named_modules():
    if hasattr(m, 'lora') and 'layers.0' in name:
        print(f"  {name}: {m.in_features}→{m.out_features}")

**解析**:

被挂 LoRA 的是 **q_proj** 和 **o_proj**(都是 768→768,方形)。

**k/v_proj 不被挂的原因**:minimind 使用 GQA(Grouped Query Attention),`num_key_value_heads=4 < num_attention_heads=8`。k/v_proj 的输出维度是 `num_kv_heads × head_dim = 4 × 96 = 384 ≠ 768`,所以不满足 `in_features == out_features`。

**如果去掉过滤,全部 Linear 都挂 LoRA:**

```python
# 假设去掉 in_features == out_features 条件
```

会多挂这些层:
- k_proj(768→384):LoRA 需要 A(768×16) + B(16×384) = 18,432 参数/层
- v_proj(768→384):同上,18,432 参数/层
- gate_proj(768→2432):A(768×16) + B(16×2432) = 51,200 参数/层
- up_proj(768→2432):同上,51,200 参数/层
- down_proj(2432→768):A(2432×16) + B(16×768) = 51,200 参数/层

总计每层多 ~190,464 参数,8 层多 ~1.5M —— LoRA 参数从 0.39M 增到 ~1.9M(占比从 0.61% 到 ~3%)。

效果可能更好(更多层参与适配),但参数量和显存增加。minimind 的方形过滤是**简洁与效果的折中**:用最少的代码挂到最关键的层。

> 工业界 LoRA(如 HuggingFace PEFT)默认会挂所有 Linear,允许用户自定义 `target_modules`。minimind 的方形过滤是一种教学友好的简化。

</details>

## Exercise 10.3（难）

**题目**:解释零初始化技巧(B=0)为什么重要。如果 B 也用高斯初始化(和 A 一样),训练开始时会发生什么?对训练收敛有什么影响?

<details><summary><b>参考答案</b></summary>

In [ ]:
import torch, torch.nn as nn

d = 768
r = 16
x = torch.randn(4, d)  # (4, 768) — 简化:用 2D 张量避免转置问题

# === 正确:A 高斯, B 零 ===
A_correct = torch.randn(r, d) * 0.02
B_correct = torch.zeros(d, r)
delta_correct = B_correct @ A_correct
print(f"=== 正确初始化 (B=0) ===")
print(f"ΔW = B@A max abs: {delta_correct.abs().max().item():.10f}")
print(f"BAx 对输出的影响: {(delta_correct @ x.T).abs().max().item():.10f}")
print(f"→ 训练开始时 LoRA 完全不影响模型输出")

# === 错误:A 高斯, B 也高斯 ===
A_wrong = torch.randn(r, d) * 0.02
B_wrong = torch.randn(d, r) * 0.02
delta_wrong = B_wrong @ A_wrong
print(f"\n=== 错误初始化 (B≠0) ===")
print(f"ΔW = B@A max abs: {delta_wrong.abs().max().item():.6f}")
print(f"BAx 对输出的影响: {(delta_wrong @ x.T).abs().max().item():.6f}")
print(f"→ 训练一开始就给输出加了一个随机扰动!")

# 测量扰动相对于原始输出的比例
W = torch.randn(d, d) * 0.02
orig_out = W @ x.T               # (768, 4) — 原始输出
perturbation = delta_wrong @ x.T  # (768, 4) — 扰动
print(f"\n原始输出 Wx 的 magnitude: {orig_out.abs().mean().item():.6f}")
print(f"扰动 ΔWx 的 magnitude:    {perturbation.abs().mean().item():.6f}")
print(f"扰动/原始 ratio:           {(perturbation.abs().mean() / orig_out.abs().mean()).item():.2%}")

**解析**:

### 为什么 B=0 很重要

LoRA 的核心假设是:在已经训练好的模型($W$ 已经很好了)上做微调,只需要一个**小的修正** $\Delta W$。

如果 $B = \mathbf{0}$:
- $\Delta W = B \cdot A = \mathbf{0}$
- 训练开始时 $W' = W + \mathbf{0} = W$ —— 模型行为和预训练模型**完全一致**
- 这意味着 LoRA 训练的起点是一个**已经收敛的好模型**,微调只是在此基础上做精细调整

### 如果 B 也用高斯初始化

$\Delta W = B \cdot A \neq \mathbf{0}$ —— 一个随机的非零矩阵。这会:

1. **立刻破坏预训练模型**:第一步 forward 就给输出加了随机噪声,模型输出和预训练模型不一致
2. **loss 暴涨**:相当于在随机模型上开始训练,需要先「恢复」到预训练水平
3. **梯度方向混乱**:初始梯度不仅要学领域知识,还要「纠正」随机初始化带来的偏差

### 数学直觉

从梯度角度看:$\frac{\partial L}{\partial B} = \frac{\partial L}{\partial (\Delta W)} \cdot A^T$

- $B = 0$ 时,梯度完全由 $A$ 和 loss 决定,方向清晰
- $B \neq 0$ 时,$\Delta W$ 已经改变了模型行为,loss landscape 变得不可预测

> 这是 LoRA 论文的核心贡献之一:不是「低秩分解」本身(这是已有技术),而是 **零初始化让低秩分解能从预训练模型平滑出发**。没有零初始化,LoRA 的效果会大幅下降。

</details>